# Methodological Framework

__TABLE OF CONTENTS__

1. [Cross-Validation](#cv)  
    1.1. [K-fold configuration](#k-fold)  
    1.2. [Shared evaluation metrics](#eval)
2. [Preprocessing Structure](#preproc)
3. [Model Assessment and Ablation Study](#ablation)
4. [Closing Summary + Github Link](#summary)

<a id="cv"></a>
## 1. Cross-Validation

To obtain a reliable estimate of out-of-sample performance and to compare models under the same conditions, all linear and regularized models in this notebook are evaluated using the same cross-validation framework. This avoids overfitting to a single train/validation split and ensures that performance comparisons between models are fair and reproducible.

<a id="k-fold"></a>
### 1.1. K-Fold configuration

We use an **8-fold K-Fold** cross-validation scheme:
- The data is split into `N_SPLITS = 8` approximately equal folds.
- At each iteration, 1 fold is used as the **validation set** and the remaining 7 folds are used as the **training set**.
- `shuffle = True` is used to randomize the split, and a fixed `RANDOM_STATE = 42` guarantees reproducibility of the folds.
- This configuration is applied consistently across all models (Linear, Ridge, Lasso, Elastic Net), so differences in performance reflect the models themselves rather than differences in the data split.

For each model and configuration, we record per-fold metrics for both **train** and **validation** sets and later aggregate them to summarize performance.

<a id="eval"></a>
### 1.2. Shared evaluation metrics

All models are observed with the same set of regression metrics:

- **RMSE (Root Mean Squared Error):** Penalizes larger errors more strongly; used as the primary measure of overall prediction error. 

  $$
  \mathrm{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}
  $$


- **MAE (Mean Absolute Error):** Measures the average absolute deviation between predictions and true values; more robust to outliers than RMSE; the one considered in Kaggle results. 

  $$
  \mathrm{MAE} = \frac{1}{n}\sum_{i=1}^{n} \lvert y_i - \hat{y}_i \rvert
  $$


- **\(R^2\) (Coefficient of Determination):** Indicates the proportion of variance in the target explained by the model. 

  $$
  R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{n}(y_i - \bar{y})^2}
  $$


- **Bias (Signed Error):**  Captures systematic overprediction (positive bias) or underprediction (negative bias).

  $$
  \text{Bias} = \frac{1}{n}\sum_{i=1}^{n} (\hat{y}_i - y_i)
  $$


For each fold we compute these metrics on both **train** and **validation** sets. Comparing train vs validation values allows us to diagnose underfitting/overfitting, while comparing across models (using mean validation metrics over folds) provides a consistent basis for model selection.

Even though we observe all these metrics for every model, **our goal is to minimize the RMSE**, and if there are two models with very similar RMSEs we then use the MAE as a tie-breaker.  
**We prioritize RMSE because, in car pricing, one massive mistake (valuing a 50k€ car at 30k€) is much worse than many tiny mistakes.** RMSE forces the model to eliminate those large, dangerous errors first, ensuring our predictions are reliable across the board. We then use MAE to refine the accuracy of 'average' cases.

<a id="preproc"></a>
## 2. Preprocessing Structure
**CONFIRMAR ESTA PARTE**

All models share a common, leakage-safe preprocessing structure to ensure fair comparison and reproducibility.

- **Deterministic cleaning:**  
    Raw data is first standardized using rule-based operations only (schema enforcement, type casting, category validation). Invalid or inconsistent values are set to `NaN`, and both identifiers (`carID`) and features that are assessed by a mechanic (`paintQuality%`) are excluded from modeling, as well as constant variables (`hasDamage`).

- **Missing values and outliers:**  
    Missing and implausible values are handled explicitly. Outliers are not removed; when necessary, extreme values are marked as missing and later imputed, preserving all observations.  

- **Imputation (CV-safe):**  
    Imputers are fitted on the training split of each CV fold and applied to validation/test data. Numeric features use robust statistics (median), while categorical features use explicit missing categories or most-frequent strategies.

- **Encoding and scaling (model-dependent):**  
    Categorical encoding and numeric scaling are applied always, even when they are not required by the model family (to make sure comparisons are consistent and coherent), always fitted on the training fold only.

- **Feature engineering and selection:**  
    Derived features and optional feature selection are treated as part of the training pipeline and executed inside cross-validation, preventing information leakage. All models are trained with and without this pipeline section — we will explain that better next.

The output of preprocessing is a fully aligned feature matrix with identical columns across train, validation, and test sets, enabling consistent evaluation and deployment.

<a id="ablation"></a>
## 3. Model Assessment and Ablation Study

To quantify the impact of specific design choices (without confounding from different CV splits or preprocessing), we benchmark **five standardized variants** across model families under the same **8-fold cross-validation** setup. In every fold, **all preprocessing, feature engineering, and feature selection steps are fit on the training split only** and then applied to the validation split (leakage-safe). Hyperparameters are explored via a **reduced random search**, reusing the same sampled configurations across folds for fair comparison. When using a log target, models are trained on `log1p(price)`, but predictions are mapped back with `expm1()` before computing RMSE/MAE/R2/bias (all reported in euros).

**The five variants are:**
1. **Baseline (raw target):** original features, no feature engineering (FE), no feature selection (FS), target = `price`.
2. **No `previousOwners` + log target:** drop `previousOwners`, target = `log1p(price)`.
3. **Age instead of year:** variant (2) + replace `year` with `age` (to represent depreciation more directly).
4. **Feature selection (80%):** variant (3) + keep the **top 80%** of features using a model-based selector (RF + `SelectFromModel`) fitted within each training fold.
5. **Full FE + FS (65%):** variant (3) + add engineered features (`owners_flagged`, mileage-based features, `engine_bin`) and retain the **top 65%** of features via the same CV-safe FS procedure.

For transparency and reproducibility, we export per-variant search results and build a compact leaderboard reporting the **best configuration per variant** (ranked by validation RMSE).

<a id="summary"></a>
## 4. Closing Summary

This notebook formalizes the methodological backbone used throughout the project to ensure that model comparisons are **fair, reproducible, and leakage-free**. We adopt a fixed **8-fold** configuration and report a shared set of regression metrics across all experiments, using **RMSE as the primary objective** (with MAE and the remaining metrics used for complementary diagnostics).

To avoid over-optimistic estimates (data-leakage), all preprocessing steps are applied under a strict **fit-on-train / transform-on-validation** rule within each fold. This includes deterministic cleaning, missing-value handling, and model-dependent encoding/scaling — always producing an aligned feature matrix across train/validation/test.

Finally, we run a controlled ablation study based on **five standardized variants** (target scaling, feature removal, age representation, and FE/FS), combined with reduced random searches to balance performance exploration and computational cost. This framework provides a consistent basis to interpret performance changes as the result of explicit modeling decisions, and it supports reliable selection of the final approach to minimize generalization error.

Our whole project is available on [INSERT LINK].